In [44]:
import networkx as nx

from complex_networks_laboratories.graph import load_graph

# Laboratorium 1

### Znaleźć źródło danych, na podstawie którego można utworzyć graf.

Mój graf - `new_zealand_collab` ([url](https://networks.skewed.de/net/new_zealand_collab))

### Co reprezentuje graf?

Graf (jako całość) reprezentuje sieć współprac pomiędzy instytutami naukowymi.
- liczba wierzchołków: 1511
- liczba krawędzi: 4273

### Co reprezentują wierzchołki?

Ww. instytuty m.in. uczelnie, organizacje. Składa się z: 
- `index` -> identyfikator wierzchołka, 
- `_graphml_vertex_id` -> nazwa instytutu, 
- `_pos` -> pozycja w UI (na ten moment nieuzywane)

### Co reprezentują krawędzie?
Krawędzie reprezentują współprace między instytutami. Składa się z:
- `source` -> wyjście krawędzi z wierzchołka źródłowego,
- `target` -> wejście krawędzi do wierzchołka docelowego
- `weight` -> waga / siła kolaboracji na podstawie ich liczby (większa waga -> więcej współprac).

Graf jest reprezentowany za pomocą macierzy symetrycznej, co czyni go nieskierowanym.

### Wczytanie do programu
Funkcja z `loader.py`, aby trzymać się konwencji utilsy/analiza.

In [45]:
graph = load_graph()

### Znalezienie najkrótszej ścieżki pomiędzy wierzchołkami wskazanymi przez użytkownika (nie uwzględniając wag)

`nx.shortest_path` korzysta z BFS (przy nie uwzględnianiu wag)

In [46]:
source = 0
target = 24
path = nx.shortest_path(graph, source, target, weight=None)
cool_str = " -> ".join(graph.nodes[node]["name"] for node in path)
print(path)
print(cool_str)

[0, 518, 24]
TS-dot Engineering Ltd. -> University of Canterbury -> Beca


### Znalezienie najkrótszej ścieżki pomiędzy wierzchołkami wskazanymi przez użytkownika (z wagami)

Zaleznie od interpretacji, moim zdaniem ciekawszym jest odwrócić wagi (1/liczba kolaboracji), aby znaleźć ściezkę po "zaprzyjaźnionych" instytutach.
`nx.shortest_path` wykorzysta algorytm Dijkstry.

In [47]:
source = 0
target = 24
for _u, _v, data in graph.edges(data=True):
    data["collaboration_distance"] = 1 / data["weight"]
path = nx.shortest_path(graph, source, target, weight="collaboration_distance")
cool_str = " -> ".join(graph.nodes[node]["name"] for node in path)
print(path)
print(cool_str)

[0, 518, 117, 24]
TS-dot Engineering Ltd. -> University of Canterbury -> University of Auckland -> Beca


### Sprawdzenie czy graf jest eulerowski 
Warunki (eulerowski):
- graf musi być spójny,
- wierzchołki o stopniu parzystym

Warunki (pół-eulerowski):
- spójny graf,
- jedynie start oraz koniec mają nieparzysty stopień wierzchołka

In [48]:
is_eulerian = nx.is_eulerian(graph)
print(is_eulerian)
is_semi_eulerian = nx.is_semieulerian(graph)
print(is_semi_eulerian)

False
False


#### Weryfikacja matematyczna ;p

In [49]:
is_connected = nx.is_connected(graph)
print(f"Czy spójny: {is_connected}")
odd_nodes_count = sum(1 for node, deg in graph.degree() if deg % 2 != 0)
print(f"Liczba wierzchołków o stopniu parzystym: {odd_nodes_count} z {len(graph.nodes())}")

Czy spójny: False
Liczba wierzchołków o stopniu parzystym: 932 z 1511


Obcinając niewielką część wierzchołków (~3%) otrzmyalibyśmy graf spójny, niemniej jednak wątpie, aby spełnił drugą regułę.

### Wyznaczenie maksymalnego przepływu pomiędzy wierzchołkami wskazanymi przez użytkownika

Zrozumienie maksymalnego przepływu bazuję na [YouTube | Max Flow Ford Fulkerson](https://www.youtube.com/watch?v=LdOnanfc5TM)

#### Maksymalny przepływ

**Po co wogóle miałbym wiedzieć co to?**

Pozwala zidentyfikować bottlenecki w grafie przepływowym (np. graf reprezentujący skrzyzowania z sygnalizacją świetlną).

**Jak przełozyć to na mój graf / dopisanie historyjki**

Dla uproszczenia przyjmuję wyimaginowane załozenie: 
- liczba wspólnych publikacji między instytutami (waga krawędzi) to przepustowość kanału – im więcej razy ze sobą publikowali,
- przepływem w sieci jest transfer wiedzy / idei badawczych (nie istnieje internet, wiedza przekazywana jest jedynie poprzez współprace)
- nie istnieją publikacje wydane samodzielnie


Przepływ na przykładzie A <-> B <-> ... <-> Y <-> Z reprezentuje: jaki jest maksymalny potencjał przekazania wiedzy z instytutu A do Z, wykorzystując całą sieć powiązań partnerskich jako pośredników.

Maksymalny przepływ wskaże nam wąskie gardło (bottleneck) – czyli które współprace są kluczowe, aby instytut Z mógł w ogóle czerpać z dorobku instytutu A.

In [50]:
source = 1
target = 388

flow_value, flow_dict = nx.maximum_flow(graph, source, target, capacity="weight")

source_name = graph.nodes[source]["name"]
target_name = graph.nodes[target]["name"]
target_capacity = graph.degree(target, weight="weight")

print(f"Potencjany udział {source_name} w publikacjach {target_name}:")
print(f"{flow_value} / {target_capacity}")

Potencjany udział HealthWEST w publikacjach Auckland Clinical Studies Ltd.:
2 / 8
